In [2]:
require(data.table)
require(tidyverse)
require(phyloseq)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)

In [64]:
ps<-readRDS(file = "/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_ITS2_outputs/RDS/ps_ITS2.rds")
#removing any taxa that don't show up in any samples to speed up the process
ps <- prune_taxa(taxa_sums(ps) > 0, ps)
ps

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 417 taxa and 475 samples ]
sample_data() Sample Data:       [ 475 samples by 24 sample variables ]
tax_table()   Taxonomy Table:    [ 417 taxa by 8 taxonomic ranks ]

In [65]:
#normalizing ps by converting rawcounts into relative abundances
#so samples with more reads wont be over represented
#using ps bc only to the count data (OTU table), while preserving the rest of the object
ps_norm = transform_sample_counts(ps, function(x) 1E6 * x / sum(x))

## vegan
- not sure if this is working with symportal data

In [66]:
# convert the sample_data() within a phyloseq object to a vegan compatible data object
pssd2veg <- function(ps_norm) {
  sample_veg<- sample_data(ps_norm)
  return(as(sample_veg,"data.frame"))
}
#using phyloseq nmds plot 
sample_veg <- pssd2veg(ps_norm)

In [67]:
# convert the otu_table() within a phyloseq object to a vegan compatible data object
psotu2veg <- function(ps_norm) {
  otu_veg <- otu_table(ps_norm)
    
  if (taxa_are_rows(otu_veg)) {
    otu_veg <- t(otu_veg)
  }
  return(as(otu_veg, "matrix"))
}

# Extract normalized OTU matrix and sample data
otu_veg <- psotu2veg(ps_norm)

In [68]:
head(otu_table(ps_norm))
head(otu_veg)

,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2027480,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851
012024_BEL_CBC_T1_557_SSID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2027480,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851
012024_BEL_CBC_T1_557_SSID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [69]:
class(otu_veg)
class(sample_veg)

[1] "matrix" "array"

[1] "data.frame"

In [70]:
head(sample_veg)

,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,Sample_physical_location,⋯,Date_ITS2,Seq_run,Transect,sample_uid,data_set_uid,host_family,host_genus,host_species,collection_date,collection_depth
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>
012024_BEL_CBC_T1_557_SSID,2_17_2025,44.6,,,12024,1_3,1/10/24,SSID,Healthy,UML_NARWHAL_R5_B18,⋯,7_7_2025,1,CBC30N,262771,1416,Rhizangiidae,Siderastrea,sidera,202401,8
012024_BEL_CBC_T1_559_MCAV,2_17_2025,14.8,,,12024,1_24,1/10/24,MCAV,Healthy,UML_NARWHAL_R5_B18,⋯,12_15_2025,2,CBC30N,267721,1458,Montastraeidae,Montastrea,cavernosa,20240110,8
012024_BEL_CBC_T1_561_OANN,2_17_2025,18,,,12024,1_25,1/10/24,OANN,Healthy,UML_NARWHAL_R5_B18,⋯,1_21_2026,3,CBC30N,267626,1457,Merulinidae,Orbicella,annularis,20240110,8
012024_BEL_CBC_T1_563_PSTR,2_17_2025,35.8,,,12024,1_12,1/10/24,PSTR,Healthy,UML_NARWHAL_R5_B18,⋯,7_7_2025,1,CBC30N,262772,1416,Faviidae,Pseudodiploria,strigosa,202401,8
012024_BEL_CBC_T1_565_PAST,2_17_2025,7.72,,,12024,1_21,1/10/24,PAST,Bleached_Tissue,UML_NARWHAL_R5_B18,⋯,2_17_2026,4,CBC30N,269104,1471,Poritidae,Porites,asteroides,20240110,8
012024_BEL_CBC_T2_585_OFAV,2_17_2025,71.6,,,12024,2_76,1/12/24,OFAV,Healthy,UML_NARWHAL_R5_B18,⋯,2_17_2026,4,SR30N,269105,1471,Merulinidae,Orbicella,faveolata,20240112,8


In [71]:
#save sammple names as a column so tidy doesn't get rid of it during filtering
sample_veg$SampleID <- rownames(sample_veg)

In [72]:
#changing format of Month_year to be Jan 2024 using collection date
sample_veg$CollectionDate <- as.factor(sample_veg$CollectionDate)
# Convert to Date format 
sample_veg$DateFormatted <- as.Date(paste0(sample_veg$CollectionDate), format = "%m/%d/%y")
# now will have a new column that has Month Year instead of date_sampled
sample_veg$Month_year <- format(sample_veg$DateFormatted, "%b %Y")
sample_veg$Month_year <- factor(sample_veg$Month_year, levels = unique(sample_veg$Month_year))

#MonthYear in chronological order
# 2. reorder MonthYear as a factor in chronological order
sample_veg$Month_year <- factor(sample_veg$Month_year,
        levels = unique(sample_veg$Month_year[order(sample_veg$DateFormatted)])
)

In [73]:
#clean sample_veg

 #cleaning up meta to only be of interest
sample_clean<- sample_veg[, c("SampleID", "Health_status", "colony", "Date_ITS2", "Condition", "Species", "Month_year", "Seq_run", "Transect", "sample_uid", "DateFormatted")]
head(sample_clean)

,SampleID,Health_status,colony,Date_ITS2,Condition,Species,Month_year,Seq_run,Transect,sample_uid,DateFormatted
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<int>,<chr>,<int>,<date>
012024_BEL_CBC_T1_557_SSID,012024_BEL_CBC_T1_557_SSID,Healthy,1_3,7_7_2025,Healthy,SSID,Jan 2024,1,CBC30N,262771,2024-01-10
012024_BEL_CBC_T1_559_MCAV,012024_BEL_CBC_T1_559_MCAV,Healthy,1_24,12_15_2025,Healthy,MCAV,Jan 2024,2,CBC30N,267721,2024-01-10
012024_BEL_CBC_T1_561_OANN,012024_BEL_CBC_T1_561_OANN,Healthy,1_25,1_21_2026,Healthy,OANN,Jan 2024,3,CBC30N,267626,2024-01-10
012024_BEL_CBC_T1_563_PSTR,012024_BEL_CBC_T1_563_PSTR,Healthy,1_12,7_7_2025,Healthy,PSTR,Jan 2024,1,CBC30N,262772,2024-01-10
012024_BEL_CBC_T1_565_PAST,012024_BEL_CBC_T1_565_PAST,Bleached_Tissue,1_21,2_17_2026,CLP,PAST,Jan 2024,4,CBC30N,269104,2024-01-10
012024_BEL_CBC_T2_585_OFAV,012024_BEL_CBC_T2_585_OFAV,Healthy,2_76,2_17_2026,Healthy,OFAV,Jan 2024,4,SR30N,269105,2024-01-12


In [74]:
otu_df <- as.data.frame(otu_veg)
#save sammple names as a column so tidy doesn't get rid of it during filtering
otu_df$SampleID <- rownames(otu_df)

### remove na from otu table

In [75]:
head(otu_df)

,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851,SampleID
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
012024_BEL_CBC_T1_557_SSID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,012024_BEL_CBC_T1_557_SSID
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_559_MCAV
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_561_OANN
012024_BEL_CBC_T1_563_PSTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,⋯,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,012024_BEL_CBC_T1_563_PSTR
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_565_PAST
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T2_585_OFAV


In [76]:
# Replace all NA values with 0
otu_df[is.na(otu_df)] <- 0

In [77]:
head(otu_df)

,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851,SampleID
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_557_SSID
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_559_MCAV
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_561_OANN
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_563_PSTR
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_565_PAST
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T2_585_OFAV


## anova using vegan
-what is driving ITS2 clustering?

In [79]:
class(otu_veg)

[1] "matrix" "array"

In [83]:
otu_veg_df <- as.data.frame(otu_veg)
# Replace all NA values with 0
otu_veg_df[is.na(otu_veg_df)] <- 0

In [84]:
otu_veg_mat <- as.matrix(otu_veg_df)

In [85]:
head(otu_veg_mat)

,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2027480,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [113]:
# remove any samples with no OTU counts
otu<- otu_veg_mat[rowSums(otu_veg_mat) > 0, ]
sample_clean <- sample_clean[rowSums(otu) > 0, ]

In [116]:
class(otu)

[1] "matrix" "array"

In [118]:
common_samples <- intersect(rownames(otu), rownames(sample_clean))

otu_clean <- otu[common_samples, ]
sample_clean <- sample_clean[common_samples, ]
all(rownames(otu_clean) == rownames(sample_clean))

[1] TRUE

In [119]:
#this one is species within transect with date as another variable
adonis2(vegdist(otu_clean, method = "bray") ~ Date_ITS2 + Month_year + Seq_run + Species + Health_status, data = sample_clean)

,Df,SumOfSqs,R2,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Date_ITS2,25,22.3954303,0.096146027,1.978814,0.001
Month_year,11,5.4636227,0.023455929,1.097169,0.019
Seq_run,1,0.4976307,0.002136383,1.099241,0.024
Species,5,10.3058698,0.044244224,4.553027,0.001
Health_status,3,1.4169321,0.006083044,1.043310,0.269
Residual,426,192.8519327,0.827934394,NA,NA
Total,471,232.9314183,1.000000000,NA,NA


## anova without vegan, otu_merged

In [88]:
head(sample_clean)

,SampleID,Health_status,colony,Date_ITS2,Condition,Species,Month_year,Seq_run,Transect,sample_uid,DateFormatted
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<int>,<chr>,<int>,<date>
012024_BEL_CBC_T1_557_SSID,012024_BEL_CBC_T1_557_SSID,Healthy,1_3,7_7_2025,Healthy,SSID,Jan 2024,1,CBC30N,262771,2024-01-10
012024_BEL_CBC_T1_559_MCAV,012024_BEL_CBC_T1_559_MCAV,Healthy,1_24,12_15_2025,Healthy,MCAV,Jan 2024,2,CBC30N,267721,2024-01-10
012024_BEL_CBC_T1_561_OANN,012024_BEL_CBC_T1_561_OANN,Healthy,1_25,1_21_2026,Healthy,OANN,Jan 2024,3,CBC30N,267626,2024-01-10
012024_BEL_CBC_T1_563_PSTR,012024_BEL_CBC_T1_563_PSTR,Healthy,1_12,7_7_2025,Healthy,PSTR,Jan 2024,1,CBC30N,262772,2024-01-10
012024_BEL_CBC_T1_565_PAST,012024_BEL_CBC_T1_565_PAST,Bleached_Tissue,1_21,2_17_2026,CLP,PAST,Jan 2024,4,CBC30N,269104,2024-01-10
012024_BEL_CBC_T2_585_OFAV,012024_BEL_CBC_T2_585_OFAV,Healthy,2_76,2_17_2026,Healthy,OFAV,Jan 2024,4,SR30N,269105,2024-01-12


In [89]:
head(otu_df)

,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,1661509,⋯,2026464,2026486,2023928,2027858,2029769,2028086,2028835,2027844,2027851,SampleID
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_557_SSID
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_559_MCAV
012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_561_OANN
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_563_PSTR
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T1_565_PAST
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,012024_BEL_CBC_T2_585_OFAV


In [90]:
otu_merged <- merge(otu_df, sample_clean, by = "SampleID",
                    all = TRUE, sort = FALSE)
head(otu_merged)

,SampleID,1661676,1661811,1661272,1661422,1661525,1661677,1661737,1669273,1661179,⋯,Health_status,colony,Date_ITS2,Condition,Species,Month_year,Seq_run,Transect,sample_uid,DateFormatted
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<int>,<chr>,<int>,<date>
1,012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,0,0,0,⋯,Healthy,1_3,7_7_2025,Healthy,SSID,Jan 2024,1,CBC30N,262771,2024-01-10
2,012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,⋯,Healthy,1_24,12_15_2025,Healthy,MCAV,Jan 2024,2,CBC30N,267721,2024-01-10
3,012024_BEL_CBC_T1_561_OANN,0,0,0,0,0,0,0,0,0,⋯,Healthy,1_25,1_21_2026,Healthy,OANN,Jan 2024,3,CBC30N,267626,2024-01-10
4,012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,⋯,Healthy,1_12,7_7_2025,Healthy,PSTR,Jan 2024,1,CBC30N,262772,2024-01-10
5,012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,⋯,Bleached_Tissue,1_21,2_17_2026,CLP,PAST,Jan 2024,4,CBC30N,269104,2024-01-10
6,012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,⋯,Healthy,2_76,2_17_2026,Healthy,OFAV,Jan 2024,4,SR30N,269105,2024-01-12


In [ ]:
 # Compute the analysis of variance
otu.anov <- aov(nonchim ~ Date_16S + factor(Month_year) + Seq_run + Species + Health_status, data = otu_merged)
# Summary of the analysis
summary(otu.anov)


## anova with taxa

ANOVA works best when:

-Each row = one observation
-Variables = columns
-The response variable is a single column (not spread across many)

In [91]:
taxa <- read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_ITS2_outputs/symportal_taxa.csv")

In [96]:
otu_long <- otu_df %>%
  pivot_longer(
    cols = -SampleID,  
    names_to = "ITS2.type.profile.UID",
    values_to = "abundance"
  )

In [97]:
head(otu_long)

SampleID,ITS2.type.profile.UID,abundance
<chr>,<chr>,<dbl>
012024_BEL_CBC_T1_557_SSID,1661676,0
012024_BEL_CBC_T1_557_SSID,1661811,0
012024_BEL_CBC_T1_557_SSID,1661272,0
012024_BEL_CBC_T1_557_SSID,1661422,0
012024_BEL_CBC_T1_557_SSID,1661525,0
012024_BEL_CBC_T1_557_SSID,1661677,0


In [101]:
str(otu_long$ITS2.type.profile.UID)
str(taxa$ITS2.type.profile.UID)

#convert taxaITS2.type.profile to chr
taxa$ITS2.type.profile.UID <- as.character(taxa$ITS2.type.profile.UID)

 chr [1:198075] "1661676" "1661811" "1661272" "1661422" "1661525" "1661677" ...
 int [1:417] 1661676 1661811 1661272 1661422 1661525 1661677 1661737 1669273 1661179 1661509 ...


In [102]:
otu_taxa <- otu_long %>%
  left_join(taxa, by = "ITS2.type.profile.UID")

In [103]:
head(otu_taxa)

SampleID,ITS2.type.profile.UID,abundance,X,Clade,Majority.ITS2.sequence,Associated.species,ITS2.profile.abundance.local,ITS2.profile.abundance.DB,ITS2.type.profile,Sequence.accession...SymPortal.UID,Average.defining.sequence.proportions.and..stdev.
<chr>,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>
012024_BEL_CBC_T1_557_SSID,1661676,0,1,A,A3/A4/A13,"Symbiodinium natans,Symbiodinium tridacnidorum,Symbiodinium necroappetens",21,26,A3/A4/A13-A4a,22385/29206/8181-40166,0.634[0.211]/0.150[0.114]-0.111[0.095]/0.105[0.186]
012024_BEL_CBC_T1_557_SSID,1661811,0,2,A,A3,"Symbiodinium natans,Symbiodinium tridacnidorum",11,230,A3,22385,1.000[0.000]
012024_BEL_CBC_T1_557_SSID,1661272,0,3,A,A3/A4,"Symbiodinium natans,Symbiodinium tridacnidorum",10,22,A3/A4/A4a,22385/29206-40166,0.657[0.295]/0.198[0.147]-0.144[0.151]
012024_BEL_CBC_T1_557_SSID,1661422,0,4,A,A3bb/A3bt,None,9,322,A3bb/A3bt,,0.515[0.062]/0.485[0.062]
012024_BEL_CBC_T1_557_SSID,1661525,0,5,A,A3/A13,"Symbiodinium natans,Symbiodinium tridacnidorum,Symbiodinium necroappetens",8,28,A3/A13,,0.627[0.220]/0.373[0.220]
012024_BEL_CBC_T1_557_SSID,1661677,0,6,A,A4/A3/A4a,"Symbiodinium natans,Symbiodinium tridacnidorum",8,8,A4/A3/A4a-A4d,29206/22385/40166-29900,0.330[0.130]/0.292[0.241]/0.288[0.136]-0.089[0.061]


In [104]:
# join sample data
full_data <- otu_taxa %>%
  left_join(sample_clean, by = "SampleID")

In [106]:
taxa.anov <- aov(abundance ~ Date_ITS2 + factor(Month_year) + Seq_run + Species + Health_status, data = full_data)
summary(taxa.anov)

                       Df    Sum Sq   Mean Sq F value Pr(>F)
Date_ITS2              25 1.393e+09 5.574e+07   0.026  1.000
factor(Month_year)     11 3.123e+08 2.839e+07   0.013  1.000
Seq_run                 1 8.138e+06 8.138e+06   0.004  0.951
Species                 5 3.863e+07 7.726e+06   0.004  1.000
Health_status           3 2.276e+07 7.588e+06   0.004  1.000
Residuals          198029 4.268e+14 2.155e+09               

In [107]:
full_data <- full_data %>%
  mutate(presence = abundance > 0)

glm_model <- glm(presence ~ Date_ITS2 + factor(Month_year) + Seq_run + Species + Health_status,
                 data = full_data,
                 family = binomial)
summary(glm_model)


Call:
glm(formula = presence ~ Date_ITS2 + factor(Month_year) + Seq_run + 
    Species + Health_status, family = binomial, data = full_data)

Deviance Residuals: 
    Min       1Q   Median       3Q      Max  
-0.1489  -0.1250  -0.1185  -0.1123   3.3609  

Coefficients:
                             Estimate Std. Error z value Pr(>|z|)   
(Intercept)                  -7.39876    2.50669  -2.952  0.00316 **
Date_ITS21_20_2026            0.01963    0.16536   0.119  0.90553   
Date_ITS21_21_2026            0.06955    0.16073   0.433  0.66525   
Date_ITS212_10_2025           0.86941    0.85758   1.014  0.31068   
Date_ITS212_15_2025           0.80206    0.86136   0.931  0.35177   
Date_ITS212_16_2025           0.85079    0.85510   0.995  0.31976   
Date_ITS212_17_2025           0.76051    0.85938   0.885  0.37618   
Date_ITS212_18_2025           0.64207    0.86112   0.746  0.45590   
Date_ITS212_19_2025           0.85242    0.85864   0.993  0.32083   
Date_ITS212_22_2025           0.22349  

#### ways to test what drives ITS2 composition
- PERMANOVA (most common)
- multivariate GLMs (mvabund)
- compositional methods

In [ ]:
 # Compute the analysis of variance
taxa.anov <- aov(nonchim ~ Date_ITS2 + factor(Month_year) + Seq_run + Species + Health_status, data = otu_merged)
# Summary of the analysis
summary(taxa.anov)


## nmds plots 